In [0]:
%pip install -U "databricks-mcp" "databricks-sdk[openai]"

In [0]:
%pip install -U \
    "mcp>=1.9" \
    "databricks-mcp" \
    "databricks-sdk[openai]"

In [0]:
dbutils.library.restartPython() 

Which Databricks workspace am I connecting to, and how do I authenticate?

Because the MCP client needs to know:
"Which Databricks workspace's MCP server should I talk to?"

In [0]:
from databricks.sdk import WorkspaceClient

workspace_client = WorkspaceClient()

print("Workspace:", workspace_client.config.host)

In [0]:
host = workspace_client.config.host

GENIE_SPACE_ID = "01f1b4e1cef91c069e5eaf691f1d8a80"

genie_mcp_url = (
    f"{host}/api/2.0/mcp/genie/{GENIE_SPACE_ID}"
)

print(genie_mcp_url)

the MCP server endpoint for that specific Genie Space.

Now create the MCP Client

In [0]:
from databricks_mcp import DatabricksMCPClient

genie_client = DatabricksMCPClient(
    server_url=genie_mcp_url,
    workspace_client=workspace_client
)

"Create an MCP client that knows how to communicate with this Genie MCP server."

In [0]:
tools = genie_client.list_tools()

In [0]:
import nest_asyncio

nest_asyncio.apply()

In [0]:
genie_tools = genie_client.list_tools()

print("Number of tools:", len(genie_tools))

In [0]:
for tool in genie_tools:
    print("Tool name:", tool.name)
    print("Description:", tool.description)
    print("Input schema:", tool.input_schema)
    print("-" * 80)

In [0]:
tool1 = genie_tools[0]

print(type(tool1))
print(dir(tool1))

In [0]:
query_tool = genie_tools[0]

print(query_tool.name)

In [0]:
result = genie_client.call_tool(
    query_tool.name,
    {
        "query": "How many customers are there?"
    }
)

print(result)

In [0]:
import databricks_mcp

print(databricks_mcp.__file__)

In [0]:
import importlib.metadata

print(importlib.metadata.version("databricks-mcp"))

In [0]:
import inspect
from databricks_mcp import DatabricksMCPClient

print(inspect.signature(DatabricksMCPClient))

In [0]:
print(genie_mcp_url)

In [0]:
import requests

response = requests.get(
    genie_mcp_url,
    timeout=30
)

print("Status:", response.status_code)
print("Response:", response.text[:1000])

In [0]:
print("Host:", workspace_client.config.host)
print("Auth type:", workspace_client.config.auth_type)

In [0]:
import inspect
import databricks_mcp.mcp as mcp_module

print(inspect.getsource(mcp_module._open_mcp_session))

In [0]:
import inspect

print(inspect.getsource(DatabricksMCPClient._call_tools_async))

In [0]:
import importlib.metadata
import httpx2

print("mcp:", importlib.metadata.version("mcp"))
print("httpx2:", httpx2.__version__)

In [0]:
import inspect
from mcp.client.streamable_http import streamable_http_client

print(inspect.signature(streamable_http_client))

In [0]:
import inspect
import httpx2

print(inspect.signature(httpx2.AsyncClient))

In [0]:
from databricks_mcp.mcp import DatabricksOAuthClientProvider

auth = DatabricksOAuthClientProvider(workspace_client)

print(type(auth))

In [0]:
import httpx2

http_client = httpx2.AsyncClient(
    auth=auth,
    follow_redirects=True,
    timeout=60.0
)

print(http_client.timeout)

In [0]:
from mcp.client.streamable_http import streamable_http_client
async with streamable_http_client(
    genie_mcp_url,
    http_client=http_client
) as (read_stream, write_stream):

    print("MCP connection opened successfully")

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async with streamable_http_client(
    genie_mcp_url,
    http_client=http_client
) as (read_stream, write_stream):

    async with ClientSession(read_stream, write_stream) as session:

        await session.initialize()

        print("MCP session initialized ✅")

        result = await session.call_tool(
            query_tool.name,
            {"query": "How many customers are there?"}
        )

        print(result)

In [0]:
poll_tool = next(
    tool for tool in genie_tools
    if tool.name.startswith("poll_response_")
)

print(poll_tool.name)

In [0]:
poll_result = await session.call_tool(
    poll_tool.name,
    {
        "conversation_id": "01f1c14546ac1f54b42face0de78c61d",
        "message_id": "01f1c14546f61424b5ebe90aeb04220f"
    }
)

print(poll_result)

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async with streamable_http_client(
    genie_mcp_url,
    http_client=http_client
) as (read_stream, write_stream):

    async with ClientSession(read_stream, write_stream) as session:

        await session.initialize()

        print("MCP session initialized ✅")

        # 1. Start Genie query
        result = await session.call_tool(
            query_tool.name,
            {"query": "How many customers are there?"}
        )

        print("Initial response:")
        print(result)

        # 2. Extract IDs
        data = result.structured_content

        conversation_id = data["conversationId"]
        message_id = data["messageId"]
        status = data["status"]

        print("\nConversation ID:", conversation_id)
        print("Message ID:", message_id)
        print("Status:", status)

        # 3. Poll while the query is still processing
        while status != "COMPLETED":

            poll_result = await session.call_tool(
                poll_tool.name,
                {
                    "conversation_id": conversation_id,
                    "message_id": message_id
                }
            )

            print("\nPoll response:")
            print(poll_result)

            data = poll_result.structured_content

            status = data["status"]

            print("Current status:", status)

            if status == "FAILED":
                print("Genie query failed.")
                break

        print("\nFinal status:", status)

In [0]:
data = result.structured_content

answer = data["content"]["textAttachments"][0]

print("Answer:", answer)

We are manually telling the MCP client:

In [0]:
from databricks.sdk import WorkspaceClient

workspace_client = WorkspaceClient()

print("Workspace:", workspace_client.config.host)
print("Auth:", workspace_client.config.auth_type)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady

workspace_client = WorkspaceClient()

print("Auth:", workspace_client.config.auth_type)

In [0]:
endpoints = list(workspace_client.serving_endpoints.list())

for ep in endpoints:
    print(ep.name)

In [0]:
from databricks.sdk import WorkspaceClient
from openai import OpenAI

workspace_client = WorkspaceClient()

llm = OpenAI(
    api_key=workspace_client.config.token,
    base_url=f"{workspace_client.config.host}/serving-endpoints"
)

print("LLM client created ✅")

In [0]:
%pip install -U databricks-openai

In [0]:
from databricks_openai import DatabricksOpenAI
from databricks.sdk import WorkspaceClient

workspace_client = WorkspaceClient()

client = DatabricksOpenAI(
    workspace_client=workspace_client
)

print("Databricks OpenAI client created ✅")

In [0]:
from databricks.sdk import WorkspaceClient

workspace_client = WorkspaceClient()

client = workspace_client.serving_endpoints.get_open_ai_client()

print("LLM client created ✅")

In [0]:
response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        {
            "role": "user",
            "content": "What is MCP in one sentence?"
        }
    ],
    max_tokens=100
)

print(response.choices[0].message.content)

In [0]:
response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        {
            "role": "system",
            "content": "In this conversation, MCP means Model Context Protocol, the protocol used to connect AI applications with tools and data sources."
        },
        {
            "role": "user",
            "content": "What is MCP?"
        }
    ],
    max_tokens=100
)

print(response.choices[0].message.content)

In [0]:
print("Tool name:")
print(query_tool.name)

print("\nTool description:")
print(query_tool.description)

print("\nTool schema:")
print(query_tool.input_schema)

In [0]:
genie_tool_definition = {
    "type": "function",
    "function": {
        "name": query_tool.name,
        "description": query_tool.description,
        "parameters": query_tool.input_schema
    }
}

print(genie_tool_definition)

In [0]:
response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful data assistant. Use the Genie tool when the user asks questions about customer or order data."
        },
        {
            "role": "user",
            "content": "How many customers are there?"
        }
    ],
    tools=[genie_tool_definition],
    tool_choice="auto",
    max_tokens=200
)

print(response.choices[0].message)

In [0]:
tool_call = response.choices[0].message.tool_calls[0]

print("Tool:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)

In [0]:
import json

arguments = json.loads(tool_call.function.arguments)

print(arguments)

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async with streamable_http_client(
    genie_mcp_url,
    http_client=http_client
) as (read_stream, write_stream):

    async with ClientSession(read_stream, write_stream) as session:

        await session.initialize()

        print("MCP session initialized ✅")

        # Execute the tool selected by the LLM
        mcp_result = await session.call_tool(
            tool_call.function.name,
            arguments
        )

        print("\nMCP result:")
        print(mcp_result)

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async with streamable_http_client(
    genie_mcp_url,
    http_client=http_client
) as (read_stream, write_stream):

    async with ClientSession(read_stream, write_stream) as session:

        await session.initialize()

        print("MCP session initialized ✅")

        # 1. Execute the tool selected by the LLM
        mcp_result = await session.call_tool(
            tool_call.function.name,
            arguments
        )

        data = mcp_result.structured_content

        conversation_id = data["conversationId"]
        message_id = data["messageId"]
        status = data["status"]

        print("Initial status:", status)

        # 2. Poll until Genie finishes
        while status != "COMPLETED":

            poll_result = await session.call_tool(
                poll_tool.name,
                {
                    "conversation_id": conversation_id,
                    "message_id": message_id
                }
            )

            data = poll_result.structured_content
            status = data["status"]

            print("Current status:", status)

            if status == "FAILED":
                print("Genie query failed.")
                break

        # 3. Extract final answer
        if status == "COMPLETED":

            answer = data["content"]["textAttachments"][0]

            print("\nFinal answer:")
            print(answer)

In [0]:
final_response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful data assistant. Answer the user's question using the tool result."
        },
        {
            "role": "user",
            "content": "How many customers are there?"
        },
        {
            "role": "assistant",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
            ]
        },
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": data["content"]["textAttachments"][0]
        }
    ],
    max_tokens=200
)

print(final_response.choices[0].message.content)